In [ ]:
import sys

sys.path.append("..")

from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import geopandas as gpd
import cartopy.crs as crs
import cartopy.feature as cfeature
from shapely.geometry import Point
import shapely.vectorized as sv
from statistics import mean

from src.data import nysm_data
from src.data import hrrr_data

In [ ]:
def date_filter(ldf, time1, time2):
    ldf = ldf[ldf["valid_time"] > time1]
    ldf = ldf[ldf["valid_time"] < time2]

    return ldf


def idw_interpolation(x, y, z, xi, yi, power=2):
    """
    x, y = known points
    z = values
    xi, yi = meshgrid of where to interpolate
    """
    dist = np.sqrt((xi[..., None] - x) ** 2 + (yi[..., None] - y) ** 2)

    # Avoid division by zero
    dist = np.where(dist == 0, 1e-12, dist)

    weights = 1 / dist**power
    z_idw = np.sum(weights * z, axis=-1) / np.sum(weights, axis=-1)
    return z_idw


def plot_nysm(nysm_df, lons, lats, values, var):
    df_ = nysm_df.copy()
    font_size = 22

    fig = plt.figure(figsize=(24, 16))
    ax = fig.add_subplot(
        1,
        1,
        1,
        projection=crs.LambertConformal(
            central_longitude=-75.0, standard_parallels=(49, 77)
        ),
    )

    # Load shapefile
    ny_state_boundaries_path = "/home/aevans/nwp_bias/src/landtype/data/State.shx"
    ny_state_boundaries_geo = gpd.read_file(ny_state_boundaries_path).to_crs(epsg=4326)

    # Get bounds for the map
    min_lon = nysm_df["lon"].min()
    max_lon = nysm_df["lon"].max()
    min_lat = nysm_df["lat"].min()
    max_lat = nysm_df["lat"].max()
    pad = 0.1

    extent = [min_lon - pad, max_lon + pad, min_lat - pad, max_lat + pad]
    ax.set_extent(extent, crs=crs.PlateCarree())

    # Gridlines
    ax.gridlines(
        crs=crs.PlateCarree(),
        draw_labels=True,
        linewidth=2,
        color="black",
        alpha=0.5,
        linestyle="--",
    )

    grid_res = 500
    grid_lon, grid_lat = np.meshgrid(
        np.linspace(min_lon - pad, max_lon + pad, grid_res),
        np.linspace(min_lat - pad, max_lat + pad, grid_res),
    )

    # IDW interpolation
    zi = idw_interpolation(x=lons, y=lats, z=values, xi=grid_lon, yi=grid_lat)

    # Mask outside NY
    ny_poly = ny_state_boundaries_geo.unary_union
    mask = ~sv.contains(ny_poly, grid_lon, grid_lat)
    zi_masked = np.ma.array(zi, mask=mask)

    # Interpolated field
    c = ax.pcolormesh(
        grid_lon,
        grid_lat,
        zi_masked,
        cmap="cool",
        transform=crs.PlateCarree(),
        shading="auto",
        # vmin=60,
        # vmax=160,
    )
    # Features
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linestyle=":", zorder=1)
    ax.add_feature(cfeature.STATES.with_scale("50m"), linestyle=":", zorder=1)
    ax.add_feature(cfeature.LAKES.with_scale("50m"), zorder=1)

    cbar = plt.colorbar(c, ax=ax, shrink=0.6)
    cbar.set_label(r"Max Accumulated Snow (in)", fontsize=font_size)
    cbar.ax.tick_params(labelsize=font_size)

    # Station scatter
    sc = ax.scatter(
        df_["lon"],
        df_["lat"],
        s=100,
        c="black",
        edgecolor="black",
        transform=crs.PlateCarree(),
        zorder=10,
    )

    # Annotate stations
    for _, row in df_.iterrows():
        ax.annotate(
            row["station"],
            (row["lon"], row["lat"]),
            textcoords="offset points",
            xytext=(0, 15),
            ha="center",
            fontsize=15,
            color="black",
            transform=crs.PlateCarree(),
            zorder=20,
        )

    # Boundary overlay
    ny_state_boundaries_geo.boundary.plot(ax=ax, edgecolor="black", linewidth=2)

    # Legend and title
    plt.title(r"NYSM: Max Snow Depth Accumulation", fontsize=font_size)

    plt.tight_layout()
    plt.show()
    # plt.savefig(
    #     "/home/aevans/nwp_bias/src/machine_learning/data/error_visuals/ALL/precip_high_impact.png"
    # )


def plot_hrrr(nysm_df, lons, lats, values, var):
    df_ = nysm_df.copy()
    font_size = 22

    fig = plt.figure(figsize=(24, 16))
    ax = fig.add_subplot(
        1,
        1,
        1,
        projection=crs.LambertConformal(
            central_longitude=-75.0, standard_parallels=(49, 77)
        ),
    )

    # Load shapefile
    ny_state_boundaries_path = "/home/aevans/nwp_bias/src/landtype/data/State.shx"
    ny_state_boundaries_geo = gpd.read_file(ny_state_boundaries_path).to_crs(epsg=4326)

    # Get bounds for the map
    min_lon = nysm_df["lon"].min()
    max_lon = nysm_df["lon"].max()
    min_lat = nysm_df["lat"].min()
    max_lat = nysm_df["lat"].max()
    pad = 0.1

    extent = [min_lon - pad, max_lon + pad, min_lat - pad, max_lat + pad]
    ax.set_extent(extent, crs=crs.PlateCarree())

    # Gridlines
    ax.gridlines(
        crs=crs.PlateCarree(),
        draw_labels=True,
        linewidth=2,
        color="black",
        alpha=0.5,
        linestyle="--",
    )

    grid_res = 500
    grid_lon, grid_lat = np.meshgrid(
        np.linspace(min_lon - pad, max_lon + pad, grid_res),
        np.linspace(min_lat - pad, max_lat + pad, grid_res),
    )

    # IDW interpolation
    zi = idw_interpolation(x=lons, y=lats, z=values, xi=grid_lon, yi=grid_lat)

    # Mask outside NY
    ny_poly = ny_state_boundaries_geo.unary_union
    mask = ~sv.contains(ny_poly, grid_lon, grid_lat)
    zi_masked = np.ma.array(zi, mask=mask)

    # Interpolated field
    c = ax.pcolormesh(
        grid_lon,
        grid_lat,
        zi_masked,
        cmap="YlGnBu",
        transform=crs.PlateCarree(),
        shading="auto",
        # vmin=0,
        # vmax=325,
    )
    # Features
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linestyle=":", zorder=1)
    ax.add_feature(cfeature.STATES.with_scale("50m"), linestyle=":", zorder=1)
    ax.add_feature(cfeature.LAKES.with_scale("50m"), zorder=1)

    cbar = plt.colorbar(c, ax=ax, shrink=0.6)
    cbar.set_label(r"Accumulated Precipitation (mm)", fontsize=font_size)
    cbar.ax.tick_params(labelsize=font_size)
    # m s

    # Station scatter
    sc = ax.scatter(
        df_["lon"],
        df_["lat"],
        s=100,
        c="black",
        edgecolor="black",
        transform=crs.PlateCarree(),
        zorder=10,
    )

    # Annotate stations
    for _, row in df_.iterrows():
        ax.annotate(
            row["station"],
            (row["lon"], row["lat"]),
            textcoords="offset points",
            xytext=(0, 15),
            ha="center",
            fontsize=15,
            color="black",
            transform=crs.PlateCarree(),
            zorder=20,
        )

    # Boundary overlay
    ny_state_boundaries_geo.boundary.plot(ax=ax, edgecolor="black", linewidth=2)

    # Legend and title
    plt.title(f"HRRR Forecast: Flash Flooding", fontsize=font_size)

    plt.tight_layout()
    plt.show()
    # plt.savefig(
    #     "/home/aevans/nwp_bias/src/machine_learning/data/error_visuals/ALL/precip_high_impact.png"
    # )


def plot_errors(nysm_df, lons, lats, values, var):
    df_ = nysm_df.copy()
    font_size = 22

    fig = plt.figure(figsize=(24, 16))
    ax = fig.add_subplot(
        1,
        1,
        1,
        projection=crs.LambertConformal(
            central_longitude=-75.0, standard_parallels=(49, 77)
        ),
    )

    # Load shapefile
    ny_state_boundaries_path = "/home/aevans/nwp_bias/src/landtype/data/State.shx"
    ny_state_boundaries_geo = gpd.read_file(ny_state_boundaries_path).to_crs(epsg=4326)

    # Get bounds for the map
    min_lon = nysm_df["lon"].min()
    max_lon = nysm_df["lon"].max()
    min_lat = nysm_df["lat"].min()
    max_lat = nysm_df["lat"].max()
    pad = 0.1

    extent = [min_lon - pad, max_lon + pad, min_lat - pad, max_lat + pad]
    ax.set_extent(extent, crs=crs.PlateCarree())

    # Gridlines
    ax.gridlines(
        crs=crs.PlateCarree(),
        draw_labels=True,
        linewidth=2,
        color="black",
        alpha=0.5,
        linestyle="--",
    )

    grid_res = 500
    grid_lon, grid_lat = np.meshgrid(
        np.linspace(min_lon - pad, max_lon + pad, grid_res),
        np.linspace(min_lat - pad, max_lat + pad, grid_res),
    )

    # IDW interpolation
    zi = idw_interpolation(x=lons, y=lats, z=values, xi=grid_lon, yi=grid_lat)

    # Mask outside NY
    ny_poly = ny_state_boundaries_geo.unary_union
    mask = ~sv.contains(ny_poly, grid_lon, grid_lat)
    zi_masked = np.ma.array(zi, mask=mask)

    # Interpolated field
    c = ax.pcolormesh(
        grid_lon,
        grid_lat,
        zi_masked,
        cmap="PiYG",
        transform=crs.PlateCarree(),
        shading="auto",
        # vmin=-250,
        # vmax=250,
    )
    # Features
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linestyle=":", zorder=1)
    ax.add_feature(cfeature.STATES.with_scale("50m"), linestyle=":", zorder=1)
    ax.add_feature(cfeature.LAKES.with_scale("50m"), zorder=1)

    cbar = plt.colorbar(c, ax=ax, shrink=0.6)
    cbar.set_label(r"Precipitation Error (mm)", fontsize=font_size)
    cbar.ax.tick_params(labelsize=font_size)
    # m s

    # Station scatter
    sc = ax.scatter(
        df_["lon"],
        df_["lat"],
        s=100,
        c="black",
        edgecolor="black",
        transform=crs.PlateCarree(),
        zorder=10,
    )

    # Annotate stations
    for _, row in df_.iterrows():
        ax.annotate(
            row["station"],
            (row["lon"], row["lat"]),
            textcoords="offset points",
            xytext=(0, 15),
            ha="center",
            fontsize=15,
            color="black",
            transform=crs.PlateCarree(),
            zorder=20,
        )

    # Boundary overlay
    ny_state_boundaries_geo.boundary.plot(ax=ax, edgecolor="black", linewidth=2)

    # Legend and title
    plt.title(r"Forecast Error (mm)", fontsize=font_size)
    # (°C)

    plt.tight_layout()
    plt.show()
    # plt.savefig(
    #     "/home/aevans/nwp_bias/src/machine_learning/data/error_visuals/ALL/precip_high_impact.png"
    # )

In [ ]:
# hrrr_df = hrrr_data.read_hrrr_data('01')

In [ ]:
def make_hrrr_data(hrrr_var, stations, time1, time2):
    hrrr_df = hrrr_data.read_hrrr_data("01")
    print(hrrr_df.columns)
    hrrr_df = hrrr_df[hrrr_df["station"].isin(stations)]
    hrrr_df = date_filter(hrrr_df, time1, time2)
    hrrr_df = hrrr_df[["valid_time", "station", hrrr_var]]

    for n in np.arange(2, 19):
        print("fh", n)
        hrrr_df_ = hrrr_data.read_hrrr_data(str(n).zfill(2))
        hrrr_df_ = hrrr_df_[hrrr_df_["station"].isin(stations)]
        hrrr_df_ = date_filter(hrrr_df_, time1, time2)
        hrrr_df_ = hrrr_df_[["valid_time", "station", hrrr_var]]
        hrrr_df = hrrr_df.merge(
            hrrr_df_,
            on=["valid_time", "station"],
            how="inner",
            suffixes=("", f"fh_{n}"),
        )

    tp_cols = [c for c in hrrr_df.columns if hrrr_var in c]
    hrrr_df[f"{hrrr_var}_avg"] = hrrr_df[tp_cols].mean(axis=1)
    hrrr_df = hrrr_df[["valid_time", "station", f"{hrrr_var}_avg"]]

    return hrrr_df

In [ ]:
def make_lstm_data(lstm_var, station, path):
    records = f'{path}/{station}'
    files = os.listdir(records)

    vals = []

    for fh in np.arange(1,19):
        read_this = [f for f in files if ('linear' in f) and (f'{lstm_var}_{fh}' in f) and ('lstm' in f)]
        lstm_df = pd.read_parquet(read_this)
        vals.append(lstm_df)

    final_df = pd.concat(vals, ignore_index=True)
    final_df["error_avg"] = final_df["Model forecast"].mean(axis=1)

    return float(final_df["error_avg"].iloc[-1])

In [ ]:
def make_hybrid_data(lstm_var, station, path):
    records = f'{path}/{station}'
    files = os.listdir(records)

    vals = []

    for fh in np.arange(1,19):
        read_this = [f for f in files if ('linear' in f) and (f'{lstm_var}_{fh}' in f) and ('hybrid' in f)]
        lstm_df = pd.read_parquet(read_this)
        vals.append(lstm_df)

    final_df = pd.concat(vals, ignore_index=True)
    final_df["error_avg"] = final_df["Model forecast"].mean(axis=1)

    return float(final_df["error_avg"].iloc[-1])

In [ ]:
def make_bnn_data(lstm_var, station, path):
    records = f'{path}/{station}'
    files = os.listdir(records)

    vals = []

    for fh in np.arange(1,19):
        read_this = [f for f in files if ('linear' in f) and (f'{lstm_var}_{fh}' in f) and ('bnn' in f)]
        lstm_df = pd.read_parquet(read_this)
        vals.append(lstm_df)

    final_df = pd.concat(vals, ignore_index=True)
    final_df["error_avg"] = final_df["Model forecast"].mean(axis=1)

    return float(final_df["error_avg"].iloc[-1])

In [ ]:
def model_average(lstm_var, station, path):
    lstm_value = make_lstm_data(lstm_var, station, path)
    hybrid_value = make_hybrid_data(lstm_var, station, path)
    bnn_value = make_bnn_data(lstm_var, station, path)

    model_avg = mean(lstm_value, hybrid_value, bnn_value)
    return model_avg

In [ ]:
def main(stations, time1, time2, var, method, hrrr_var):
    # load nysm data
    nysm_df = nysm_data.load_nysm_data(gfs=False)
    nysm_df = nysm_df.rename(columns={"time_1H": "valid_time"})
    print(nysm_df.columns)
    # print(nysm_df)

    # filter for effected stations
    nysm_df = nysm_df[nysm_df["station"].isin(stations)]

    # filter for time
    nysm_df = date_filter(nysm_df, time1, time2)
    # print(nysm_df)

    # hrrr_data
    hrrr_df = make_hrrr_data(hrrr_var, stations, time1, time2)
    # print(hrrr_df)

    if method == "accumulate":
        print("accumulating")
        nysm_df = nysm_df.sort_values(["station", "valid_time"])
        hrrr_df = hrrr_df.sort_values(["station", "valid_time"])
        nysm_df[f"{var}_t"] = nysm_df.groupby("station")[var].cumsum()
        hrrr_df[f"{hrrr_var}_t"] = hrrr_df.groupby("station")[
            f"{hrrr_var}_avg"
        ].cumsum()

    nysm_df.dropna(inplace=True)
    values = []
    hrrr_vals = []
    # model_vals = []
    lats = []
    lons = []

    for s in nysm_df["station"].unique():
        df_ = nysm_df[nysm_df["station"] == s]
        hrrr_temp = hrrr_df[hrrr_df["station"] == s]
        # model_out = model_average(var, s, path)
        # model_vals.append(model_out)

        if method == "accumulate":
            values.append(df_[f"{var}_t"].iloc[-1])
            hrrr_vals.append(hrrr_temp[f"{hrrr_var}_t"].iloc[-1])
            lats.append(df_["lat"].iloc[-1])
            lons.append(df_["lon"].iloc[-1])
        if method == "max":
            values.append(df_[f"{var}"].max())
            hrrr_vals.append(hrrr_temp[f"{hrrr_var}_avg"].max())
            lats.append(df_["lat"].iloc[-1])
            lons.append(df_["lon"].iloc[-1])
        if method == "min":
            values.append(df_[f"{var}"].min())
            hrrr_vals.append(hrrr_temp[f"{hrrr_var}_avg"].min())
            lats.append(df_["lat"].iloc[-1])
            lons.append(df_["lon"].iloc[-1])

    # Convert lists → arrays
    values = np.array(values, dtype=float)
    hrrr_vals = np.array(hrrr_vals, dtype=float)
    lats = np.array(lats, dtype=float)
    lons = np.array(lons, dtype=float)
    # error = nwp - nysm
    errors = hrrr_vals - values
    # model_err = model_vals - errors
    # Print mean error ignoring NaN
    mean_err = np.nanmean(errors)
    q1 = np.nanpercentile(errors, 25)
    q3 = np.nanpercentile(errors, 75)

    print("Mean error:", mean_err)
    print("Q1 (25th percentile):", q1)
    print("Q3 (75th percentile):", q3)

    # plots
    # print(nysm_df, lons, lats, values, var)
    plot_nysm(nysm_df, lons, lats, values, var)
    plot_hrrr(nysm_df, lons, lats, hrrr_vals, var)
    plot_errors(nysm_df, lons, lats, errors, var)

In [ ]:
time1 = datetime(2024, 8, 8, 0, 0, 0)
time2 = datetime(2024, 8, 10, 23, 59, 59)
nysm_var = "precip_total"
hrrr_var = "tp"
method = "accumulate"

nysm_clim = pd.read_csv("/home/aevans/nwp_bias/src/landtype/data/nysm.csv")

## whole nysm
# stations = nysm_clim['stid'].unique()

# one division
c = "Champlain Valley"
nysm_ = nysm_clim[nysm_clim["climate_division_name"] == c]

# # # selection of divisions
# use_ls = [
#     "Great Lakes",
#     "Western Plateau",
#     "Northern Plateau",
#     "St. Lawrence Valley",
#     "Central Lakes",
# ]
# nysm_ = nysm_clim[nysm_clim["climate_division_name"].isin(use_ls)]

stations = nysm_["stid"].unique()

main(stations, time1, time2, nysm_var, method, hrrr_var)

# Plot Case Study

# WeatherBench